<a href="https://colab.research.google.com/github/lianplass/lianplass.github.io/blob/master/IPUMS_API_Guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieving Data from IPUMS via API
This notebook walks you through how to query the IPUMS API using the `ipumspy` Python library to retrieve individual-level microdata from IPUMS USA and spatially referenced aggregate data from NHGIS.  

**Learning Objectives**
- Extract census data
- Load it into a dataframe
- Join it to county geometries
- Produce a choropleth map of the result
---

## Part I.- Microdata Extraction (IPUMS USA)
In this section you will be extracting microdata from IPUMS. For reference, **Microdata** refers to individual-level records — each row represents one person or one household. IPUMS USA provides harmonized microdata from the U.S. Census and American Community Survey (ACS), making it possible to compare variables consistently across decades.
<br>

>**Why Microdata and Not Aggregate Data?**

> Lets say you want to find out whether `elderly people` `living alone` are more likely to have `no health insurance`. You might have a hard time finding the exact table that corresponds to this data, so in this case you'll want to use individual-level records (i.e.,one row per person) to cross-tabulate multiple characteristics at once.  These characteristics are:
> - Is this person over 65?
> - Do they live in a single-person household?
> - Do they have health insurance?

> None of this is possible with pre-aggregated data (e.g., census tract, census block group aggregated data), because the aggregation destroys the relationships between variables at the individual level. For instance, you can't know from a county summary table whether the uninsured people in a county are also the elderly people living alone — the data has already been collapsed.


### 1. Environment Setup: Install Dependencies, Import Libraries

Begin by installing the `ipumspy` library. This library is the official Python client for the IPUMS API.  The following line installds it into you notebook's runtime environment.

In [ ]:
# Install ipumspy
!pip install ipumspy

Import the following dependencies.  These are the recommended dependencies in the library's documentation.

These imports bring in the core tools you'll use throughout:

- `IpumsApiClient` — authenticates your session and handles API requests
- `MicrodataExtract` — defines the parameters for an individual-level data extract
-`readers` and `ddi` — handle reading the downloaded data files into Python

In [ ]:
# Import necessary dependencies from Getting Started guide
import os
from pathlib import Path
from ipumspy import IpumsApiClient, MicrodataExtract, readers, ddi

### 2. Generate Your API Key
`ipumspy` documentation notes, "to interact with the IPUMS API, you'll need to register for access with the IPUMS project you'll be using."

**[Click Here to Create an Account and Retrieve an API Key](https://account.ipums.org/authentication/login?return_url=https%3A%2F%2Faccount.ipums.org%2Fapi_keys)**

> 🚨 **Note:** This is an extremely important step.  If you do not complete it you will only encounter errors from here on out.

API secrets should be kept secret! Therefore you will be loading in the IPUMS API key from Colab's built-in Secrets manager.

Find the key icon in the left sidebar and create a new key called `IPUMS_API_KEY`.

By storing your key in the manager instead of pasting it directly into the notebook you can keep your sensitive access credentials out of the code, even if you share the notebook.

### 3. Load Key into Environment
Using the code below, load the key into your active runtime environment

> 🚨 **Note:** You may need to do this twice for the subsequent NHGIS data extraction.  If you encounter an error later on review the error message and follow the appropriate steps to amend it.

In [ ]:
'''
Create and add secret for API access to Colab using the sidebar then
load into active runtime environment
'''
from google.colab import userdata
ipums = IpumsApiClient(userdata.get('IPUMS_API_KEY'))

### 4. Construct your Query Variable

The `extract` variable in the code below  defines what you want to extract, but doesn't submit it yet. The key parameters:

- `collection` - the IPUMS dataset to query. "usa" refers to IPUMS USA (decennial census + ACS)
- `description` - a label that appears in your IPUMS account's extract history
samples — which survey sample(s) to include. - `"us2012b"` is the 2012 ACS 5-year sample. You can find sample codes in the IPUMS USA documentation
- `variables` - which variables to include. Here we're pulling age and sex. See the full variable list in the IPUMS codebook

In [ ]:
# Define extraction variable (this will need to be edited later on...)
extract = MicrodataExtract(
    collection="usa",
    description="Sample USA extract",
    samples=["us2012b"],
    variables=["AGE","SEX"]
)

### 5.  Submit, Wait, and Download
Once you have defined the variable, run the code below to submit your query.  The query will store the data locally or in your VM's storage.

For microdata extracts, IPUMS returns two files: a compressed `.dat.gz` data file and an `.xml` DDI codebook that describes the variables.

Be sure to review and edit the DOWNLOAD_DIR path definition to ensure the data is downloading to the correct location.

In [ ]:
ipums.submit_extract(extract)
ipums.wait_for_extract(extract)

DOWNLOAD_DIR = Path("/content")
ipums.download_extract(extract, download_dir=DOWNLOAD_DIR)

### 6. Explore the Structure of the Downloaded Data

To explore the data that has just been downloaded, we'll port it over into a Pandas DataFrame.

IPUMS data uses a fixed-width format described by a DDI (Data Documentation Initiative) codebook. The `read_ipums_ddi()` function parses the .xml codebook, and `read_microdata()` uses it to correctly parse the `.dat.gz` file into a pandas DataFrame.  The function will apply human readable variable labels and value codes as well.

In [ ]:
# Load the downloaded data into runtime environment no need to extract independently
ddi = readers.read_ipums_ddi("/content/usa_00005.xml")
ipums_df = readers.read_microdata(ddi, "/content/usa_00005.dat.gz")


Use the code cell below to preview your DataFrame.

In [ ]:
# Print the first five values of the extracted data
print(ipums_df.head())

### 7. Understanding Structure of Microdata Extracts
In addition to values for `AGE` and `SEX` the resulting DataFrame contains a set of technical variables (e.g., identifiers, weights, and clustering info).  These are necessary for statistical analysis.

**Here are some other notable characteristics of your microdata extract DataFrame that affect usage:**
- Each row is a person, but people are nested within households. The `SERIAL` column is the household ID, and `PERNUM` identifies each person within that household (1 = first person, 2 = second, etc.). This lets you link household-level context to individual records if needed.

- Coded integer values, not labels. SEX will appear as 1 and 2, not "Male" and "Female". IPUMS uses integer codes for categorical variables. The DDI codebook contains the lookup table for every code, and ipumspy stores this metadata on the DataFrame — but the cell values themselves remain numeric. You can find the codes for any variable in the [IPUMS USA online codebook](https://usa.ipums.org/usa-action/variables/group).

- Sampling weights matter. If you're computing statistics (means, proportions, counts), you should use PERWT as your weight — it tells you how many people in the real population each record represents. A raw unweighted row count from this DataFrame is not a reliable population estimate.


## ❗️ Try it Out ❗️
Define and submit your own IPUMS USA microdata extract using a different sample year and a different set of variables.

Extract individual-level data for two variables (e.g., RACE, EDUC, TRANWORK) of your choosing from the 2019 ACS 1-year sample(`us2019a`), then load it into a DataFrame and preview the first five rows.

When you have finished, answer the Comprehension Check questions in the code cell.

> **Hint:** The structure of MicrodataExtract is identical to what you used in Part I.  You will just need to swap in a new sample code and variable list. The full list of available variables and sample codes can be browsed in the **[IPUMS USA codebook](https://usa.ipums.org/usa-action/variables/group)**. The IPUMS USA codebook is your reference for browsing available variables by theme, checking which sample years each variable covers, and looking up the integer codes that represent categorical values in your DataFrame. Always check the Codes tab for any categorical variable before doing analysis.

In [ ]:
# Your code here


# -------------------------------------------------------
# COMPREHENSION CHECK: Answer these questions in a comment below
# 1. How many columns does your DataFrame have?
# 2. What dtype are your two chosen variables?
# 3. Are your variables coded as integers or strings?
#    What do the codes mean? (check the IPUMS codebook)
# -------------------------------------------------------

# Your answers:
# 1.
# 2.
# 3.

---
## Part II.- Downloading Spatially Referenced Aggregate Data from NHGIS

This part of the lab exercise will involve extrating aggregate spatially referenced data from IPUMS NHGIS.

What is aggregate data? Unlike microdata, aggregate data is pre-summarized to a geographic unit (e.g., county, tract, state). NHGIS provides aggregate census data paired with boundary shapefiles, making it ideal for mapping and spatial analysis.

### 1. Environment Setup
Begin by importing `AggregateDataExtract` and `NhgisDataset`. These classes are structured to handle NHGIS's dataset/table/geography hierarchy.

> 🚨 **Note:** You will use the shutil import later to extract compressed data.  You can run the command now or later in Step 4 of this part.

> 🚨 **Note:** You will use the geopandas import later to handle your spatially referenced aggregate data.  You can run the command now or later in Step 7 of this part.

In [ ]:
from ipumspy import AggregateDataExtract, NhgisDataset

# import shutil
# import geopandas as gpd

### 2. Define an NHGIS Tabular Data Extract
Once again, you will be redefining your `extract` variable to faciliate extraction.  You can use the template code below and the [NHGIS Data Finder](https://data2.nhgis.org/main) to do this.

In [ ]:
extract = AggregateDataExtract(
    collection="nhgis",
    description="CYPLAN101 Lab NHGIS extract",
    datasets=[
        NhgisDataset(name="1990_STF1",data_tables=["NP1","NP2"], geog_levels=["county"])
    ]
)

###3. Submit, Wait, and Download the Tabular Extract
Repeat this sequence of steps from the microdata extract workflow. The output from this process will be a `.zip` archive containing a CSV and a codebook file.

In [ ]:
ipums.submit_extract(extract)
ipums.wait_for_extract(extract)

DOWNLOAD_DIR = Path("/content")
ipums.download_extract(extract, download_dir=DOWNLOAD_DIR)

###4. Unpack and Load the CSV into a DataFrame
NHGIS delivers tabular data as a zipped CSV. shutil.unpack_archive() extracts it, and pd.read_csv() loads it into a DataFrame.

The encoding='latin-1' argument is necessary because NHGIS CSVs use Latin-1 encoding rather than UTF-8.

The resulting DataFrame has one row per county and columns for each requested summary table variable.

In [ ]:
# extract the csv.zip then convert to df; if you have not already, import shutil
nhgisCSV = shutil.unpack_archive("/content/nhgis0001_csv.zip", "/content/nhgis0001_csv")

nhgisDF = pd.read_csv(f"/content/nhgis0001_csv/nhgis0001_ds120_1990_county.csv", encoding='latin-1')

print(nhgisDF.head())

Preview the result using the code cell below.

In [ ]:
print(ipums_df_02.head())

###5. Retrieve Corresponding Geometry
Begin by defining a shapefile extract.  

Separate from the tabular extract, NHGIS also provides boundary shapefiles which can be merged with the tabular data into a GeoDataFrame (GDF). The extract definition in the sample request below will retrieve  the 1990 county boundary file (derived from TIGER/Line 2000 geometries). The string `"us_county_1990_tl2000"` is the NHGIS shapefile code — you can find codes in the NHGIS data finder under the "GIS Files" tab.

In [ ]:
extract = AggregateDataExtract(
   collection="nhgis",
   shapefiles=["us_county_1990_tl2000"]
)

###6. Submit, Wait, and Download the Shapefile Extract
Once again, you will repeat the extract workflow.  This time you will retrieve a `.zip` file containing the shapefile components (e.g., `.shp`,`.dbf`,`shx`)

In [ ]:
# Submit extraction using API credentials and download zip and xml file
ipums.submit_extract(extract)

ipums.wait_for_extract(extract)

DOWNLOAD_DIR = Path("/content")
ipums.download_extract(extract, download_dir=DOWNLOAD_DIR)

###7. Load the Shapefile into a GeoDataFrame
GeoPandas reads the shapefile directly from its nested .zip archive into a GeoDataFrame.

The CRS (coordinate reference system) is read automatically from the .prj file.

In [ ]:
shutil.unpack_archive("/content/nhgis0003_shape.zip", "/content/nhgis0003_shape")
nhgisGDF = gpd.read_file("/content/nhgis0003_shape/nhgis0003_shape/nhgis0003_shapefile_tl2000_us_county_1990.zip")

nhgisGDF.head()


Using the code below, confirm that the appropriate data has been brought in from your query.

In [ ]:
nhgisGDF.plot()

###8. Merge the Tabular Data and Geometries
If your aim is to analyze or visualize particular demographic data, it will be helpful to combine the aggregate attribute data retrieved in the early steps of this part of the API guide with the feature data you've just retrieved.  

IPUMS makes this relatively easy by including a common `GISJOIN` field in both datasets. Running a merge on this key joins each county's census variables to its geometry, producing a single GeoDataFrame ready for mapping.

To do this, run the code cell below.

In [ ]:
nhgisMerged = nhgisGDF.merge(nhgisDF, on='GISJOIN')
print(nhgisMerged.head())

Confirm that the resulting GDF contains the expected columns. Use this to identify the exact variable codes (e.g., ET1001 for total population) that you'll reference when plotting.

In [ ]:
print(nhgisMerged.columns.tolist())

###9. Optional: Produce a Choropleth Map of the Total Population by County for the US in 1990
Run the code cell below to create a choropleth map of your data (dependencies included inline)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15,10))

nhgisMerged.plot(
    column='ET1001',
    ax=ax,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Total Population', 'orientation': 'horizontal'},
    edgecolor='black',
    linewidth=0.3,
    missing_kwds={'color': 'lightgrey'},
)

ax.set_title("Total Pop by County, 1990")
ax.set_axis_off()

plt.tight_layout()
plt.show()

## ❗️ Try it Out ❗️
Define and submit your own NHGIS aggregate data extract using a different census dataset and geographic level.

Extract county-level population data from the **2000 Census** (`2000_SF1b`), load it into a DataFrame, merge it with the corresponding county shapefile (`us_county_2000_tl2000`), and produce a choropleth map of a variable of your choosing.

When you have finished, answer the Comprehension Check questions in the code cell.

> **Hint:** The structure of `AggregateDataExtract` and `NhgisDataset` is identical to what you used in Part II. You will just need to swap in the new dataset name and a table of your choosing. The full list of available datasets, data tables, and geographic levels can be browsed in the **[NHGIS Data Finder](https://data2.nhgis.org/main)**. Once you've loaded your merged GeoDataFrame, use `print(nhgisMerged.columns.tolist())` to identify the column name for the variable you want to map — NHGIS column names are coded (e.g., `FXS001`) and the codebook CSV included in your extract download contains the lookup.


In [ ]:
# Your code here


# -------------------------------------------------------
# COMPREHENSION CHECK: Answer these questions in a comment below
# 1. What is the key column used to merge the tabular data and shapefile, and why does NHGIS use it instead of county name or FIPS code?
# 2. How many rows does your merged GeoDataFrame have, and what does each row represent?
# 3. What variable did you map, and what spatial pattern do you observe?
# -------------------------------------------------------

# Your answers:
# 1.
# 2.
# 3.

---

## Attribution & Acknowledgments

This guide was created by **Li Plass**, University of California, Berkeley, 2026.

If you use or adapt this material, please provide attribution.

**Data Source:** All data retrieved through this guide is sourced from [IPUMS](https://www.ipums.org), University of Minnesota. When publishing or presenting work that uses IPUMS data, you are required to cite the appropriate dataset. Citation formats for each collection can be found at [ipums.org/citation](https://www.ipums.org/citation).

**Use of AI Assistance:** Portions of this guide such as draft explanatory text, code scaffolding, and instructional framing were developed with the assistance of Claude Sonnet. All content has been reviewed, edited, and validated by the author. All pedagogical decisions and interpretations are the author's own.

---